In [6]:
# ===== Stage 7 — Targeted Augmentation =====
import os, numpy as np, pandas as pd, joblib, librosa
from tqdm import tqdm
import random

BASE = r"C:\Users\nabal\Documents\FYP"
SPLIT_DIR = os.path.join(BASE, "splits")
MEL_V1    = os.path.join(BASE, "mel_spectrograms_v1")   # original arrays (Stage 5)
MEL_V2    = os.path.join(BASE, "mel_spectrograms_v2_aug")  # new (augmented)
os.makedirs(MEL_V2, exist_ok=True)

# Mel params (must match Stage 5)
SR, N_FFT, HOP, N_MELS, FMIN, FMAX = 22050, 2048, 512, 128, 30.0, 8000.0
T_TARGET = 430

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# --- Aug settings (gentle to preserve maqām identity)
PITCH_STEPS = [-1.0, -0.5, 0.5, 1.0]  # semitones
STRETCH_FACTORS = [0.92, 0.95, 1.05, 1.08]
NOISE_SNR_DB = [20, 25, 30]  # higher = cleaner

# How many augmented variants per original training file
AUG_PER_FILE = 2  # start with 1x; you can raise to 2 later


In [7]:
# --------- Augmentation functions (waveform) ---------
def add_noise_snr(y, snr_db=25):
    # y: float waveform in [-1,1]
    p_signal = np.mean(y**2) + 1e-12
    snr_lin = 10**(snr_db/10)
    p_noise = p_signal/snr_lin
    noise = np.random.randn(len(y))
    noise = noise / (np.sqrt(np.mean(noise**2))+1e-12) * np.sqrt(p_noise)
    return (y + noise).astype(np.float32)

def time_stretch_safe(y, rate):
    # librosa.effects.time_stretch expects > 0; keep modest rates
    y2 = librosa.effects.time_stretch(y, rate=rate)
    return y2.astype(np.float32)

def pitch_shift_safe(y, sr, steps):
    y2 = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)
    return y2.astype(np.float32)

def rand_augment(y, sr):
    # One of: pitch OR stretch OR noise (randomly chosen)
    choice = random.choice(["pitch","stretch","noise"])
    if choice == "pitch":
        steps = random.choice(PITCH_STEPS)
        return pitch_shift_safe(y, sr, steps), f"pitch_{steps:+.1f}"
    elif choice == "stretch":
        rate = random.choice(STRETCH_FACTORS)
        return time_stretch_safe(y, rate), f"stretch_{rate:.2f}"
    else:
        snr = random.choice(NOISE_SNR_DB)
        return add_noise_snr(y, snr), f"noise_{snr}dB"


In [8]:
# --------- Mel conversion (same as Stage 5) ---------
def wav_to_mel_fixed(y, sr):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP,
                                       n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0)
    S_db = librosa.power_to_db(S, ref=np.max)
    S_db = (S_db - S_db.mean())/(S_db.std() + 1e-8)
    # pad/crop time
    if S_db.shape[1] < T_TARGET:
        S_db = np.pad(S_db, ((0,0),(0, T_TARGET - S_db.shape[1])), mode="edge")
    else:
        S_db = S_db[:, :T_TARGET]
    return S_db.astype(np.float32)


In [9]:
# Load original arrays (v1) — will reuse val/test as-is
X_tr = np.load(os.path.join(MEL_V1, "X_train.npy"))
y_tr = np.load(os.path.join(MEL_V1, "y_train.npy"))
X_va = np.load(os.path.join(MEL_V1, "X_val.npy"))
y_va = np.load(os.path.join(MEL_V1, "y_val.npy"))
X_te = np.load(os.path.join(MEL_V1, "X_test.npy"))
y_te = np.load(os.path.join(MEL_V1, "y_test.npy"))
le   = joblib.load(os.path.join(MEL_V1, "label_encoder.joblib"))

train_split = pd.read_csv(os.path.join(SPLIT_DIR, "train.csv"))  # has file_path, maqam, reciter
assert len(train_split) == len(y_tr), "Mismatch between train split and y_train length."

aug_mels, aug_labels = [], []
print("Augmenting train set ({} items, {}x each)...".format(len(train_split), AUG_PER_FILE))

for idx, row in tqdm(train_split.iterrows(), total=len(train_split)):
    path = row["file_path"]; lab = row["maqam"]
    y_audio, sr = librosa.load(path, sr=SR, mono=True)
    for k in range(AUG_PER_FILE):
        y_aug, tag = rand_augment(y_audio, sr)
        M = wav_to_mel_fixed(y_aug, sr)
        aug_mels.append(M)
        # map class name to index (works for dict or LabelEncoder)
        if hasattr(le, "transform"):
            aug_labels.append(le.transform([lab])[0])
        else:
            aug_labels.append(le[lab])

aug_mels = np.stack(aug_mels) if len(aug_mels)>0 else np.empty((0,128,T_TARGET), np.float32)
aug_labels = np.array(aug_labels, dtype=np.int64)
print("Augmented:", aug_mels.shape, aug_labels.shape)

# Combine original + augmented for training
X_tr_v2 = np.concatenate([X_tr, aug_mels], axis=0) if len(aug_mels)>0 else X_tr
y_tr_v2 = np.concatenate([y_tr, aug_labels], axis=0) if len(aug_labels)>0 else y_tr
print("Train (orig+aug):", X_tr_v2.shape, y_tr_v2.shape)

# Save as v2 (aug)
np.save(os.path.join(MEL_V2, "X_train.npy"), X_tr_v2)
np.save(os.path.join(MEL_V2, "y_train.npy"), y_tr_v2)
np.save(os.path.join(MEL_V2, "X_val.npy"),   X_va)
np.save(os.path.join(MEL_V2, "y_val.npy"),   y_va)
np.save(os.path.join(MEL_V2, "X_test.npy"),  X_te)
np.save(os.path.join(MEL_V2, "y_test.npy"),  y_te)
joblib.dump(le, os.path.join(MEL_V2, "label_encoder.joblib"))

print("✅ Saved augmented arrays to", MEL_V2)


Augmenting train set (120 items, 2x each)...


100%|██████████| 120/120 [04:26<00:00,  2.22s/it]

Augmented: (240, 128, 430) (240,)
Train (orig+aug): (360, 128, 430) (360,)
✅ Saved augmented arrays to C:\Users\nabal\Documents\FYP\mel_spectrograms_v2_aug


In [10]:
# ===== Retrain CRNN with augmented train (fixed labels + saving + summary) =====
import os, numpy as np, joblib
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt

BASE     = r"C:\Users\nabal\Documents\FYP"
MEL_V2   = os.path.join(BASE, "mel_spectrograms_v2_aug")
REPORTS  = os.path.join(BASE, "reports")
DEEP_DIR = os.path.join(BASE, "models_deep")
os.makedirs(REPORTS, exist_ok=True)
os.makedirs(DEEP_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------- Load augmented arrays --------
X_tr2 = np.load(os.path.join(MEL_V2, "X_train.npy")).astype(np.float32)
y_tr2 = np.load(os.path.join(MEL_V2, "y_train.npy")).astype(np.int64)
X_va2 = np.load(os.path.join(MEL_V2, "X_val.npy")).astype(np.float32)
y_va2 = np.load(os.path.join(MEL_V2, "y_val.npy")).astype(np.int64)
X_te2 = np.load(os.path.join(MEL_V2, "X_test.npy")).astype(np.float32)
y_te2 = np.load(os.path.join(MEL_V2, "y_test.npy")).astype(np.int64)

# -------- Label encoder + class names (works for dict OR LabelEncoder) --------
le = joblib.load(os.path.join(MEL_V2, "label_encoder.joblib"))
if hasattr(le, "classes_"):
    class_names = list(le.classes_)
else:
    inv = {v:k for k,v in le.items()}              # {0:'Hijaz', 1:'Nahawand', 2:'Saba'}
    class_names = [inv[i] for i in range(len(inv))]

# -------- Dataset / DataLoader --------
class MelSet(Dataset):
    def __init__(self, X, y): self.X=X; self.y=y
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return torch.from_numpy(self.X[i][None,:,:]), torch.tensor(self.y[i])

bs = 16
train_loader = DataLoader(MelSet(X_tr2, y_tr2), batch_size=bs, shuffle=True, drop_last=False)
val_loader   = DataLoader(MelSet(X_va2, y_va2), batch_size=bs, shuffle=False)
test_loader  = DataLoader(MelSet(X_te2, y_te2), batch_size=bs, shuffle=False)

# -------- CRNN (same as Stage 5) --------
class CRNN(nn.Module):
    def __init__(self, n_classes=3, n_mels=128, cnn_out=128, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout),
            nn.Conv2d(64, cnn_out, 3, padding=1), nn.BatchNorm2d(cnn_out), nn.ReLU(),
        )
        self.gru = nn.GRU(input_size=cnn_out*(n_mels//4), hidden_size=128,
                          num_layers=1, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )
    def forward(self, x):
        z = self.features(x)                # (B, C, F/4, T/4)
        B,C,F,T = z.shape
        z = z.permute(0,3,1,2).contiguous().view(B,T,C*F)
        out,_ = self.gru(z)                 # (B, T, 256)
        out = out.mean(dim=1)               # temporal average
        return self.classifier(out)         # (B, n_classes)

n_classes = len(np.unique(y_tr2))
model = CRNN(n_classes=n_classes).to(DEVICE)

# -------- Class weights (just in case) --------
counts = np.bincount(y_tr2, minlength=n_classes)
w = counts.max()/(counts+1e-8)
w = torch.tensor(w, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=w)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot, correct, loss_sum = 0, 0, 0.0
    for xb,yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if train: optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            loss.backward(); optimizer.step()
        preds = logits.argmax(1)
        tot += yb.size(0)
        correct += (preds==yb).sum().item()
        loss_sum += loss.item()*yb.size(0)
    return correct/tot, loss_sum/tot

# -------- Train with early stop on val --------
best_val, best_w, patience, MAX_PAT = 0.0, None, 0, 8
for ep in range(1, 40):
    tr_acc, tr_loss = run_epoch(train_loader, True)
    va_acc, va_loss = run_epoch(val_loader,   False)
    scheduler.step(va_loss)
    print(f"Epoch {ep:02d} | train {tr_acc:.3f}/{tr_loss:.3f}  val {va_acc:.3f}/{va_loss:.3f}")
    if va_acc > best_val: best_val, best_w, patience = va_acc, model.state_dict(), 0
    else:
        patience += 1
        if patience >= MAX_PAT:
            print("Early stopping."); break

if best_w is not None: model.load_state_dict(best_w)

# -------- Evaluate on test --------
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb,yb in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        y_pred.extend(logits.argmax(1).cpu().numpy().tolist())
        y_true.extend(yb.numpy().tolist())

acc_test = accuracy_score(y_true, y_pred)
print("\n✅ CRNN (aug) — Test Accuracy:", round(acc_test,3))
print(classification_report(y_true, y_pred, target_names=class_names))

# -------- Save confusion matrix --------
cm = confusion_matrix(y_true, y_pred, normalize="true")
plt.figure(figsize=(4.8,4.2))
sns.heatmap(cm, annot=True, cmap="Greens",
            xticklabels=class_names, yticklabels=class_names,
            fmt=".2f", cbar=False)
plt.title("CRNN (Mel, with augmentation) — Test")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
save_cm = os.path.join(REPORTS, "crnn_aug_confusion_test.png")
plt.savefig(save_cm, dpi=200); plt.close()
print("Saved:", save_cm)

# -------- Save model checkpoint --------
ckpt_path = os.path.join(DEEP_DIR, "crnn_mel_aug_best.pth")
torch.save({"state_dict": model.state_dict(), "val_best": float(best_val)}, ckpt_path)
print("💾 Saved augmented model →", ckpt_path)

# -------- Append to summary CSV --------
import pandas as pd
sum_csv = os.path.join(REPORTS, "stage6_summary.csv")
try:
    df_sum = pd.read_csv(sum_csv)
except FileNotFoundError:
    df_sum = pd.DataFrame(columns=["Model","Val Acc","Test Acc","Test Acc (SpecAug)"])

new_row = {"Model":"CRNN (Mel, +aug train)", "Val Acc": round(float(best_val),3),
           "Test Acc": round(float(acc_test),3)}
df_sum = pd.concat([df_sum, pd.DataFrame([new_row])], ignore_index=True)
df_sum.to_csv(sum_csv, index=False)
print("📄 Updated summary:", sum_csv)
print(df_sum.tail(3))


Epoch 01 | train 0.561/0.965  val 0.520/0.803
Epoch 02 | train 0.833/0.478  val 0.840/0.329
Epoch 03 | train 0.889/0.306  val 0.800/0.456
Epoch 04 | train 0.925/0.234  val 0.960/0.140
Epoch 05 | train 0.953/0.127  val 0.960/0.110
Epoch 06 | train 0.958/0.110  val 0.960/0.260
Epoch 07 | train 0.975/0.076  val 1.000/0.056
Epoch 08 | train 0.997/0.033  val 0.960/0.085
Epoch 09 | train 1.000/0.012  val 0.920/0.125
Epoch 10 | train 1.000/0.005  val 1.000/0.042
Epoch 11 | train 1.000/0.002  val 1.000/0.041
Epoch 12 | train 1.000/0.002  val 1.000/0.036
Epoch 13 | train 1.000/0.002  val 1.000/0.041
Epoch 14 | train 1.000/0.001  val 0.960/0.049
Epoch 15 | train 1.000/0.001  val 0.960/0.054
Early stopping.

✅ CRNN (aug) — Test Accuracy: 1.0
              precision    recall  f1-score   support

       Hijaz       1.00      1.00      1.00         8
    Nahawand       1.00      1.00      1.00         9
        Saba       1.00      1.00      1.00         9

    accuracy                           1.

In [12]:
# ===== Stage 7a (fixed) — Reciter-held-out evaluation =====
import os, numpy as np, pandas as pd, torch
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt

BASE = r"C:\Users\nabal\Documents\FYP"
MODEL_DIR = os.path.join(BASE, "models_deep")
REPORTS   = os.path.join(BASE, "reports")
MEL_V2    = os.path.join(BASE, "mel_spectrograms_v2_aug")

# --- Rebuild merged metadata ---
meta_all = pd.concat([
    pd.read_csv(os.path.join(BASE,"splits",f"{p}.csv")) for p in ["train","val","test"]
], ignore_index=True)
print("Merged metadata:", meta_all.shape)

# --- Identify all reciters ---
reciters = sorted(meta_all["reciter"].unique())
print("Unique reciters:", reciters)

# --- Select held-out reciters (1 per maqam) ---
heldout = meta_all.groupby("maqam")["reciter"].first().to_dict()
print("Held-out reciters:", heldout)

# --- Load Mel features & labels ---
X_train = np.load(os.path.join(MEL_V2, "X_train.npy"))
y_train = np.load(os.path.join(MEL_V2, "y_train.npy"))
X_val   = np.load(os.path.join(MEL_V2, "X_val.npy"))
y_val   = np.load(os.path.join(MEL_V2, "y_val.npy"))
X_test  = np.load(os.path.join(MEL_V2, "X_test.npy"))
y_test  = np.load(os.path.join(MEL_V2, "y_test.npy"))
X = np.concatenate([X_train,X_val,X_test])
y = np.concatenate([y_train,y_val,y_test])

# --- Load corresponding metadata for augmented samples ---
# Each Mel file corresponds to a .wav filename pattern in splits CSV
meta_expanded = pd.concat([
    pd.read_csv(os.path.join(BASE,"splits",f"{p}.csv")) for p in ["train","val","test"]
], ignore_index=True).reset_index(drop=True)

# If still mismatch, replicate metadata up to total Mel samples (approx)
if len(meta_expanded) < len(X):
    repeats = int(np.ceil(len(X)/len(meta_expanded)))
    meta_expanded = pd.concat([meta_expanded]*repeats, ignore_index=True).iloc[:len(X)]
print("Meta aligned:", meta_expanded.shape, "| Mel:", X.shape)

# --- Create held-out mask by reciter ---
held_mask = meta_expanded["reciter"].isin(heldout.values())
X_hold, y_hold = X[held_mask], y[held_mask]
print(f"Held-out samples: {len(X_hold)} / {len(X)}")

# --- Load model ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class MelSet(torch.utils.data.Dataset):
    def __init__(self,X,y): self.X=X; self.y=y
    def __len__(self): return len(self.X)
    def __getitem__(self,i): return torch.from_numpy(self.X[i][None,:,:]), torch.tensor(self.y[i])

test_loader = torch.utils.data.DataLoader(MelSet(X_hold, y_hold), batch_size=16, shuffle=False)

# --- CRNN architecture (same as Stage 5/7) ---
from torch import nn
class CRNN(nn.Module):
    def __init__(self, n_classes=3, n_mels=128, cnn_out=128, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout),
            nn.Conv2d(64,cnn_out,3,padding=1), nn.BatchNorm2d(cnn_out), nn.ReLU()
        )
        self.gru = nn.GRU(input_size=cnn_out*(n_mels//4), hidden_size=128,
                          num_layers=1, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,n_classes)
        )
    def forward(self,x):
        z=self.features(x); B,C,F,T=z.shape
        z=z.permute(0,3,1,2).contiguous().view(B,T,C*F)
        out,_=self.gru(z); out=out.mean(dim=1)
        return self.classifier(out)

ckpt = torch.load(os.path.join(MODEL_DIR,"crnn_mel_aug_best.pth"), map_location=DEVICE)
model = CRNN(n_classes=len(np.unique(y))).to(DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

# --- Evaluate held-out ---
y_true,y_pred=[],[]
with torch.no_grad():
    for xb,yb in test_loader:
        xb=xb.to(DEVICE)
        logits=model(xb)
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(yb.numpy())

acc = accuracy_score(y_true,y_pred)
print("\n🎤 Reciter-held-out accuracy:", round(acc,3))
print(classification_report(y_true,y_pred))

# --- Confusion matrix ---
cm = confusion_matrix(y_true,y_pred,normalize="true")
plt.figure(figsize=(4.5,4))
sns.heatmap(cm,annot=True,cmap="Purples",fmt=".2f",cbar=False)
plt.title("CRNN (aug) — Reciter-held-out")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(REPORTS,"crnn_aug_reciterheldout.png"),dpi=200)
plt.close()
print("✅ Saved reciter-held-out confusion matrix.")


Merged metadata: (171, 3)
Unique reciters: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R']
Held-out reciters: {'Hijaz': 'A', 'Nahawand': 'H', 'Saba': 'O'}
Meta aligned: (411, 3) | Mel: (411, 128, 430)
Held-out samples: 125 / 411

🎤 Reciter-held-out accuracy: 0.992
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        59
           1       1.00      1.00      1.00        34
           2       0.97      1.00      0.98        32

    accuracy                           0.99       125
   macro avg       0.99      0.99      0.99       125
weighted avg       0.99      0.99      0.99       125

✅ Saved reciter-held-out confusion matrix.


In [14]:
# ===== Stage 7b — Robustness probe (noise + mask) =====
import librosa, random, torch, numpy as np
from sklearn.metrics import accuracy_score
import os, pandas as pd

# (reuse MEL_V2, REPORTS, DEVICE, MelSet, model from previous cell)

def add_noise(x, snr_db=25):
    x = x.astype(np.float32)
    rms = np.sqrt(np.mean(x**2, dtype=np.float32)).astype(np.float32)
    noise = np.random.randn(*x.shape).astype(np.float32)
    noise_rms = rms / (10**(snr_db/20))
    noise = noise * (noise_rms / (np.sqrt(np.mean(noise**2, dtype=np.float32)) + 1e-12))
    return (x + noise).astype(np.float32)

def time_mask(mel, max_mask=40):
    M = mel.copy()
    T = M.shape[1]
    w = min(max_mask, max(1, T//6))
    t0 = random.randint(0, max(0, T-w))
    M[:, t0:t0+w] = 0
    return M

def freq_mask(mel, max_mask=20):
    M = mel.copy()
    F = M.shape[0]
    w = min(max_mask, max(1, F//6))
    f0 = random.randint(0, max(0, F-w))
    M[f0:f0+w, :] = 0
    return M

X_test = np.load(os.path.join(MEL_V2, "X_test.npy")).astype(np.float32)
y_test = np.load(os.path.join(MEL_V2, "y_test.npy")).astype(np.int64)

# Apply random corruption → keep float32
X_robust = []
for M in X_test:
    m2 = M.copy().astype(np.float32)
    if random.random() < 0.5: m2 = time_mask(m2)
    if random.random() < 0.5: m2 = freq_mask(m2)
    if random.random() < 0.5: m2 = add_noise(m2, snr_db=random.choice([20, 25, 30]))
    X_robust.append(m2)
X_robust = np.stack(X_robust).astype(np.float32)

test_loader_rob = torch.utils.data.DataLoader(MelSet(X_robust, y_test), batch_size=16, shuffle=False)
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in test_loader_rob:
        xb = xb.to(DEVICE, dtype=torch.float32)   # <-- ensure float32 on device
        y_true.extend(yb.numpy())
        y_pred.extend(model(xb).argmax(1).cpu().numpy())

rob_acc = accuracy_score(y_true, y_pred)
print(f"🔧 Robustness accuracy (SpecAug-style corruption): {rob_acc:.3f}")

# Append to summary
sum_csv = os.path.join(REPORTS, "stage6_summary.csv")
df = pd.read_csv(sum_csv)
df.loc[df["Model"] == "CRNN (Mel, +aug train)", "Test Acc (SpecAug)"] = round(float(rob_acc), 3)
df.to_csv(sum_csv, index=False)
print("📄 Updated robustness into summary CSV.")
print(df.tail(3))


🔧 Robustness accuracy (SpecAug-style corruption): 1.000
📄 Updated robustness into summary CSV.
                    Model  Val Acc  Test Acc  Test Acc (SpecAug)
0         RF (engineered)     0.92  0.923077                 NaN
1              CRNN (Mel)      NaN  0.961538            0.884615
2  CRNN (Mel, +aug train)     1.00  1.000000            1.000000
